In [2]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np

# ==============================================================================
# CELL: STRATEGY 1 - MERGE GZT BACKGROUND DATA WITH RZLT TAX DATA
# ==============================================================================
print("[1/4] Loading RZLT and GZT datasets...")

# 1. Load RZLT Positives (Class 1)
rzlt_path = "Datasets/RZLT_Samples/rzlt_sample_CWLA.geojson"  # Replace with your RZLT file path
gdf_rzlt = gpd.read_file(rzlt_path)
gdf_rzlt['LIVE_SITE'] = 'Y'

gzt_path = "Datasets/GZT_Samples/CWLA_GZT_Sample.geojson"

gdf_gzt = gpd.read_file(gzt_path)

# Ensure both layers use Irish Transverse Mercator Projection (EPSG:2157)
if gdf_rzlt.crs != "EPSG:2157":
    gdf_rzlt = gdf_rzlt.to_crs(epsg=2157)
if gdf_gzt.crs != "EPSG:2157":
    gdf_gzt = gdf_gzt.to_crs(epsg=2157)

print(f"Loaded {len(gdf_rzlt)} RZLT parcels ('Y').")
print(f"Loaded {len(gdf_gzt)} GZT parcels.")

[1/4] Loading RZLT and GZT datasets...
Loaded 1341 RZLT parcels ('Y').
Loaded 200 GZT parcels.


In [3]:
# ==============================================================================
# [2/4] SPATIAL OVERLAY DIFFERENCE (GZT MINUS RZLT)
# ==============================================================================
print("[2/4] Performing spatial overlay difference to extract non-taxed land...")

# Filter GZT for statutory zones suitable for development (e.g. R2, R1, M2, C6)
# Exclude purely open-space or unzoned codes if desired
taxable_gzt_codes = ['R1', 'R2', 'C6', 'M2', 'M3', 'C2.1']
gdf_gzt_eligible = gdf_gzt[gdf_gzt['ZONE_GZT'].isin(taxable_gzt_codes)].copy()

# Perform Spatial Difference: isolate zoned land NOT covered by RZLT polygons
gdf_non_taxed = gpd.overlay(gdf_gzt_eligible, gdf_rzlt, how='difference')
gdf_non_taxed['LIVE_SITE'] = 'N'

# Compute area in hectares and filter out tiny slivers (< 0.01 ha)
gdf_non_taxed['SITE_AREA'] = gdf_non_taxed.geometry.area / 10000.0
gdf_non_taxed = gdf_non_taxed[gdf_non_taxed['SITE_AREA'] >= 0.01].copy()

print(f"Extracted {len(gdf_non_taxed)} candidate non-taxed parcels ('N').")

[2/4] Performing spatial overlay difference to extract non-taxed land...
Extracted 51 candidate non-taxed parcels ('N').


C:\Users\musab\AppData\Roaming\Python\Python313\site-packages\geopandas\tools\overlay.py:386: UserWarning: `keep_geom_type=True` in overlay resulted in 7 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)


In [ ]:
# ==============================================================================
# [3/4] SCHEMA ALIGNMENT & DOWNSAMPLING
# ==============================================================================
print("[3/4] Aligning columns and balancing class ratios...")

# Recalculate SITE_AREA for RZLT parcels if missing/unstandardized
if 'SITE_AREA' not in gdf_rzlt.columns or gdf_rzlt['SITE_AREA'].isnull().any():
    gdf_rzlt['SITE_AREA'] = gdf_rzlt.geometry.area / 10000.0

# Standardize Local Authority identifier
gdf_rzlt['LOCAL_AUTHORITY'] = 'CWLA'
gdf_non_taxed['LOCAL_AUTHORITY'] = 'CWLA'

# Keep matching core feature columns
core_cols = ['geometry', 'LIVE_SITE', 'SITE_AREA', 'ZONE_GZT', 'LOCAL_AUTHORITY']

rzlt_subset = gdf_rzlt[[c for c in core_cols if c in gdf_rzlt.columns]].copy()
non_taxed_subset = gdf_non_taxed[[c for c in core_cols if c in gdf_non_taxed.columns]].copy()

# Balance dataset: Sample Class 'N' to achieve 1:1 or max 1:2 ratio against Class 'Y'
num_positives = len(rzlt_subset)
non_taxed_balanced = non_taxed_subset.sample(
    n=min(len(non_taxed_subset), num_positives * 2),  # up to 2x the Y count
    random_state=42
)

# Concatenate into unified classification dataframe
gdf_combined = pd.concat([rzlt_subset, non_taxed_balanced], ignore_index=True)

# Encode binary target (Y -> 1, N -> 0)
gdf_combined['target'] = (gdf_combined['LIVE_SITE'] == 'Y').astype(int)

# Re-extract ITM spatial centroids for KDTree transit lookups
gdf_combined['centroid_x'] = gdf_combined.geometry.centroid.x
gdf_combined['centroid_y'] = gdf_combined.geometry.centroid.y

[3/4] Aligning columns and balancing class ratios...


In [6]:
# ==============================================================================
# [4/4] EXPORT MERGED STRATEGY 1 DATASET
# ==============================================================================
output_path = "Datasets/RZLT_GZT_Merged/CWLA_Merged.geojson"

gdf_combined.to_file(output_path, driver="GeoJSON")

print("\n=== Dataset Merge Complete ===")
print(f"Total Rows: {len(gdf_combined)}")
print("Class Distribution:")
print(gdf_combined['LIVE_SITE'].value_counts())
print(f"\nMerged dataset successfully saved to '{output_path}'!")


=== Dataset Merge Complete ===
Total Rows: 1392
Class Distribution:
LIVE_SITE
Y    1341
N      51
Name: count, dtype: int64

Merged dataset successfully saved to 'Datasets/RZLT_GZT_Merged/CWLA_Merged.geojson'!
